# Building tool-using agents with LangChain

This notebook introduces LangChain agents by building a small shopping assistant. The assistant can look up product prices and retrieve product descriptions and categories, choosing which tools to call for each request.

You will learn how to:

1. Define typed tools with the `@tool` decorator.
2. Create an agent with a model, tools, and a system prompt.
3. Invoke the agent and inspect its model and tool messages.
4. Stream intermediate agent steps.
5. Preserve conversation state with a LangGraph checkpointer.

The product catalog is local and deterministic. Only the chat model call requires network access and an OpenAI API key.

## Environment setup

Choose a model with `MODEL_NAME`, then load the matching provider credential from the project's local `.env` file. OpenAI uses `OPENAI_API_KEY`, Groq uses `GROQ_API_KEY`, and Google uses `GOOGLE_API_KEY`.

In [1]:
import os
import sys

from dotenv import load_dotenv


load_dotenv()

MODEL_NAME = "llama-3.3-70b-versatile"
SUPPORTED_MODELS = {
    "llama-3.3-70b-versatile": "GROQ_API_KEY",
    "qwen3-32b": "GROQ_API_KEY",
    "gpt-4.1-mini": "OPENAI_API_KEY",
    "gemma-4-31b-it": "GOOGLE_API_KEY",
}

if MODEL_NAME not in SUPPORTED_MODELS:
    supported = ", ".join(SUPPORTED_MODELS)
    raise ValueError(f"MODEL_NAME must be one of: {supported}")

required_api_key = SUPPORTED_MODELS[MODEL_NAME]
assert os.getenv(required_api_key), f"Add {required_api_key} to your .env file."

print(f"Python executable: {sys.executable}")
print(f"Selected model: {MODEL_NAME}")

Python executable: /Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/bin/python3
Selected model: llama-3.3-70b-versatile


## Define tools

A tool is a Python function exposed to the model with a name, description, and input schema. LangChain derives the schema from the function's type hints and uses the docstring to tell the model when the tool is useful.

These tools deliberately expose separate views of the same nested product catalog. The agent chooses the price tool for cost questions and the product-details tool for category or description questions.

In [2]:
from langchain.tools import tool


PRODUCT_CATALOG = {
    "laptop stand": {
        "price": 54.99,
        "category": "electronics",
        "description": "An adjustable aluminum stand for laptops up to 17 inches.",
    },
    "mechanical keyboard": {
        "price": 89.50,
        "category": "electronics",
        "description": "A compact mechanical keyboard with tactile switches.",
    },
    "wireless mouse": {
        "price": 34.25,
        "category": "electronics",
        "description": "An ergonomic wireless mouse with adjustable sensitivity.",
    },
    "garden hose": {
        "price": 42.00,
        "category": "garden",
        "description": "A kink-resistant 50-foot hose for everyday garden watering.",
    },
    "pruning shears": {
        "price": 24.75,
        "category": "garden",
        "description": "Bypass pruning shears for trimming stems and small branches.",
    },
    "electric kettle": {
        "price": 49.99,
        "category": "home appliances",
        "description": "A 1.7-liter electric kettle with automatic shutoff.",
    },
    "air purifier": {
        "price": 129.00,
        "category": "home appliances",
        "description": "A compact HEPA air purifier designed for medium-sized rooms.",
    },
}


@tool
def get_product_price(product_name: str) -> str:
    """Look up the price of one product in the store catalog."""
    normalized_name = product_name.strip().lower()
    product = PRODUCT_CATALOG.get(normalized_name)
    if product is None:
        available = ", ".join(sorted(PRODUCT_CATALOG))
        return f"Product not found. Available products: {available}."
    return f"{normalized_name}: ${product['price']:.2f}"


@tool
def get_product_details(product_name: str) -> str:
    """Look up a product's category and description in the store catalog."""
    normalized_name = product_name.strip().lower()
    product = PRODUCT_CATALOG.get(normalized_name)
    if product is None:
        available = ", ".join(sorted(PRODUCT_CATALOG))
        return f"Product not found. Available products: {available}."
    return (
        f"{normalized_name} | category: {product['category']} | "
        f"description: {product['description']}"
    )


tools = [get_product_price, get_product_details]

/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Inspect the generated tool schemas

The model sees these schemas rather than the Python implementation. Clear names, type hints, parameter names, and docstrings improve tool selection and argument generation.

In [3]:
for current_tool in tools:
    print(f"Tool: {current_tool.name}")
    print(f"Description: {current_tool.description}")
    print(f"Input schema: {current_tool.args}")
    print()

Tool: get_product_price
Description: Look up the price of one product in the store catalog.
Input schema: {'product_name': {'title': 'Product Name', 'type': 'string'}}

Tool: get_product_details
Description: Look up a product's category and description in the store catalog.
Input schema: {'product_name': {'title': 'Product Name', 'type': 'string'}}



## Create the agent

`create_agent` builds a model-and-tools loop on top of LangGraph. The model can answer directly or request a tool call. After a tool runs, its result is added to the message history and the model decides whether another tool is needed.

In [4]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI


if MODEL_NAME == "gpt-4.1-mini":
    model = ChatOpenAI(model=MODEL_NAME, temperature=0)
elif MODEL_NAME == "gemma-4-31b-it":
    model = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0)
else:
    groq_model_name = {
        "llama-3.3-70b-versatile": "llama-3.3-70b-versatile",
        "qwen3-32b": "qwen/qwen3-32b",
    }[MODEL_NAME]
    model = ChatGroq(
        model=groq_model_name,
        temperature=0,
        reasoning_format="hidden" if MODEL_NAME == "qwen3-32b" else None,
    )

print(f"Model integration: {type(model).__name__}")

SYSTEM_PROMPT = """You are a precise shopping assistant.
Use get_product_price for every catalog price question.
Use get_product_details for every category or description question.
Use both tools when the user asks for both kinds of information.
Do not invent product information that was not returned by a tool.
Answer concisely and clearly identify the product.
"""

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

Model integration: ChatGroq


## Invoke the agent

Agent input and output are state dictionaries containing a `messages` sequence. This request should cause two tool calls because it asks for both price and descriptive product metadata.

In [5]:
request = {
    "messages": [
        {
            "role": "user",
            "content": (
                "What is the electric kettle's price, category, and description?"
            ),
        }
    ]
}

result = agent.invoke(request)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The electric kettle's price is $49.99, it belongs to the category of home appliances, and its description is a 1.7-liter electric kettle with automatic shutoff.


## Inspect the agent trace

The returned state includes the complete message sequence. `AIMessage` objects can contain requested tool calls, and each `ToolMessage` records a tool result linked to the corresponding call ID.

In [6]:
from langchain_core.messages import AIMessage, ToolMessage


for index, message in enumerate(result["messages"], start=1):
    print(f"{index}. {message.type}")

    if isinstance(message, AIMessage) and message.tool_calls:
        for tool_call in message.tool_calls:
            print(f"   requested: {tool_call['name']}({tool_call['args']})")
    elif isinstance(message, ToolMessage):
        print(f"   returned: {message.content}")
    elif message.content:
        print(f"   content: {message.content}")

1. human
   content: What is the electric kettle's price, category, and description?
2. ai
   requested: get_product_price({'product_name': 'electric kettle'})
   requested: get_product_details({'product_name': 'electric kettle'})
3. tool
   returned: electric kettle: $49.99
4. tool
   returned: electric kettle | category: home appliances | description: A 1.7-liter electric kettle with automatic shutoff.
5. ai
   content: The electric kettle's price is $49.99, it belongs to the category of home appliances, and its description is a 1.7-liter electric kettle with automatic shutoff.


## Stream intermediate steps

For longer tasks, streaming exposes progress before the final answer is ready. With `stream_mode="updates"`, each event contains the state update produced by a graph node such as `model` or `tools`.

In [7]:
stream_request = {
    "messages": [
        {
            "role": "user",
            "content": (
                "Tell me the price, category, and description of the garden hose."
            ),
        }
    ]
}

for update in agent.stream(stream_request, stream_mode="updates"):
    node_name, node_update = next(iter(update.items()))
    latest_message = node_update["messages"][-1]
    print(f"Node: {node_name}")

    if isinstance(latest_message, AIMessage) and latest_message.tool_calls:
        called_tools = [call["name"] for call in latest_message.tool_calls]
        print(f"Tool request: {called_tools}")
    else:
        print(latest_message.content)
    print()

Node: model
Tool request: ['get_product_price', 'get_product_details']

Node: tools
garden hose: $42.00

Node: tools
garden hose | category: garden | description: A kink-resistant 50-foot hose for everyday garden watering.



Node: model
The price of the garden hose is $42.00, it belongs to the garden category, and its description is: A kink-resistant 50-foot hose for everyday garden watering.



## Add short-term memory

Agents are stateless unless a checkpointer stores their state. `InMemorySaver` is suitable for local examples and tests. A `thread_id` identifies one conversation; reusing it lets a later invocation access earlier messages.

In production, use a persistent checkpointer rather than in-memory storage.

In [8]:
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver


memory_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid4())}}

first_turn = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My budget is $80. Remember that for my next question.",
            }
        ]
    },
    config=config,
)
first_turn["messages"][-1].pretty_print()

================================== Ai Message ==================================

I've taken note of your budget of $80 for our next conversation.


In [9]:
follow_up = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Would the electric kettle fit my budget, and what is it used for?"
                ),
            }
        ]
    },
    config=config,
)
follow_up["messages"][-1].pretty_print()

================================== Ai Message ==================================

The electric kettle costs $49.99 and is used for boiling water. It fits your budget of $80.


## Key takeaways

- Agents combine a chat model, tools, instructions, and an execution loop.
- Tool schemas and descriptions are part of the agent's interface and should be designed carefully.
- The returned message history makes model decisions and tool results inspectable.
- Streaming helps surface progress during multi-step work.
- Checkpointers preserve conversation state when the same `thread_id` is reused.

Good production agents also need tool error handling, authorization boundaries, observability, evaluation, and human approval for consequential actions.

## Possible next steps

The following features are useful extensions, but are intentionally left out to keep this introductory single-agent example focused:

- **Structured output:** Validate the final response with a Pydantic model so downstream code receives predictable fields and types.
- **Error handling:** Add explicit handling for unknown products, invalid tool arguments, provider errors, timeouts, and retryable failures.
- **Agent evaluation:** Add assertions or test cases that verify the selected tools, tool arguments, final answer, and behavior across each supported model.
- **Tool-call limits:** Stop the agent after a configured number of steps to prevent accidental loops and control cost.
- **LangSmith tracing:** Inspect model latency, token usage, tool calls, errors, and complete execution traces.
- **Async execution:** Demonstrate `ainvoke()` and `astream()` for applications that handle concurrent requests.
- **Runtime context:** Pass user preferences, locale, or currency without placing application data directly in the prompt.
- **Human approval:** Pause before tools perform consequential or irreversible actions.

Structured output, error handling, and basic evaluation are the strongest candidates for the next iteration. Human-in-the-loop workflows, middleware, durable persistence, and production observability are better suited to separate advanced-agent tutorials.